# NFL_GAME_AWAY_POINTS

Workspace notebook on Container Runtime (SPCS). Same contract as `ml/`:
`label_away_points` only, train 2023–24 RS+post, walk-forward 2025, no `close_*` in X,
`log_model` to `NFL_PROD_DB.ML` with `pip_requirements` and SPCS only.

Away score. Same matchup X as game total.

SQL cells inspect. Python cells fit, register, and score via `weekend_warriors_ml`.
Keep that package in this Workspace (the `ml/` folder from the repo).

Run the install cell first. Extra `uv pip` installs do not survive weekend service restarts.
One-cell path at the bottom: `cook(session, SPEC.name)`.

## Packages

`uv pip` against the Snowflake-managed PyPI artifact repo (no Anaconda).
Rerun this after a notebook-service restart. If it cannot reach a repo, attach
the Snowflake PyPI artifact repository on the notebook service — do not point
this at public pypi.org.

In [ ]:
!uv pip install scikit-learn snowflake-ml-python

import importlib.metadata as md

print("sklearn", md.version("scikit-learn"))
print("snowflake-ml-python", md.version("snowflake-ml-python"))

## FEATURES grain

Expect 1,323 games. Eligible RS+post completed should be 285 per season 2023–25.

In [ ]:
%%sql -r matchup_grain
SELECT
    COUNT(*) AS rows,
    COUNT(IFF(is_completed AND season_type IN (2, 3), 1, NULL)) AS eligible_rs_post,
    COUNT(IFF(label_away_points IS NULL, 1, NULL)) AS label_null,
    COUNT(IFF(close_implied_away_total IS NOT NULL, 1, NULL)) AS close_nonnull
FROM NFL_PROD_DB.FEATURES.FEAT_GAME_MATCHUP

In [ ]:
%%sql -r eligible_by_season
SELECT season, COUNT(*) AS n
FROM NFL_PROD_DB.FEATURES.FEAT_GAME_MATCHUP
WHERE is_completed AND season_type IN (2, 3) AND label_away_points IS NOT NULL
GROUP BY 1
ORDER BY 1

## Session and import

Kernel session via `get_active_session()`. Add `ml/` to `sys.path` so this
notebook can import the same module the laptop `uv run` uses.

In [ ]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for cand in (here, here.parent, here.parent.parent):
    if (cand / "weekend_warriors_ml" / "pipeline.py").exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
else:
    raise FileNotFoundError(
        "weekend_warriors_ml not found. Put the repo ml/ folder in this Workspace."
    )

from snowflake.snowpark.context import get_active_session
from weekend_warriors_ml.specs import get_spec
from weekend_warriors_ml.pipeline import (
    cook,
    fit,
    inspect,
    log_experiment,
    register,
    score_batch,
    score_local,
)

SPEC = get_spec("NFL_GAME_AWAY_POINTS")
session = get_active_session()
print(SPEC.name, SPEC.task, len(SPEC.feature_columns), "features")
print(session.get_current_role(), session.get_current_warehouse())

In [ ]:
inspect(session, SPEC.name)

## Fit

HistGradientBoostingRegressor. Prints error vs a mean baseline.

In [ ]:
model, metrics = fit(session, SPEC.name)
metrics

## Register

Logs the experiment (best-effort), then `log_model` to `NFL_PROD_DB.ML` for SPCS.
`version_name` is whatever `log_model` auto-generates.

In [ ]:
try:
    log_experiment(session, metrics, SPEC.name)
except Exception as exc:
    print("Experiment tracking skipped:", exc)

version_name = register(session, model, SPEC.name)
version_name

In [ ]:
%%sql -r models
SHOW MODELS LIKE 'NFL_GAME_AWAY_POINTS' IN SCHEMA NFL_PROD_DB.ML

In [ ]:
%%sql -r model_fns
SHOW FUNCTIONS IN MODEL NFL_PROD_DB.ML.NFL_GAME_AWAY_POINTS VERSION {{version_name}}

## Score

Writes `NFL_PROD_DB.ML.PRED_GAME_AWAY_POINTS` from the in-kernel model (full slate).

In [ ]:
preds = score_local(session, model, version_name, SPEC.name)
preds.head(5)

In [ ]:
%%sql -r pred_summary
SELECT
    COUNT(*) AS rows,
    COUNT(label_away_points) AS labeled,
    COUNT(close_implied_away_total) AS with_close,
    MIN(scored_at) AS scored_at,
    MIN(version_name) AS version_name
FROM NFL_PROD_DB.ML.PRED_GAME_AWAY_POINTS

In [ ]:
%%sql -r walkforward
SELECT
    COUNT(*) AS n_2025_rs_post,
    ROUND(AVG(ABS(pred_away_points - label_away_points)), 3) AS mae,
    ROUND(AVG(POW(pred_away_points - label_away_points, 2)), 3) AS mse
FROM NFL_PROD_DB.ML.PRED_GAME_AWAY_POINTS
WHERE season = 2025
  AND season_type IN (2, 3)
  AND label_away_points IS NOT NULL

## Optional: `run_batch` on ML_DEV_POOL

Skip if you only needed the pred table. Pool is `CPU_X64_S`, not `DLT_POOL`.

In [ ]:
score_batch(session, version_name, SPEC.name)

## One cell

Same as inspect + fit + register + score_local. Does not fire the pool.
Use this if an agent is driving the notebook end to end.

In [ ]:
cook(session, SPEC.name)